# Steam Games Analysis


In [ ]:
import os
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

os.makedirs('charts', exist_ok=True)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

ESTIMATED_OWNERS_MAP = {
    '0 - 20000': '0 - 20K',
    '20000 - 50000': '20K - 50K',
    '50000 - 100000': '50K - 100K',
    '100000 - 200000': '100K - 200K',
    '200000 - 500000': '200K - 500K',
    '500000 - 1000000': '500K - 1M',
    '1000000 - 2000000': '1M - 2M',
    '2000000 - 5000000': '2M - 5M',
    '5000000 - 10000000': '5M - 10M',
    '10000000 - 20000000': '10M - 20M',
    '20000000 - 50000000': '20M - 50M',
    '50000000 - 100000000': '50M - 100M',
    '100000000 - 200000000': '100M - 200M'
}

ESTIMATED_OWNERS_ORDER = [
    '0 - 20K', '20K - 50K', '50K - 100K', '100K - 200K', '200K - 500K',
    '500K - 1M', '1M - 2M', '2M - 5M', '5M - 10M', '10M - 20M',
    '20M - 50M', '50M - 100M', '100M - 200M'
]

POPULAR_THRESHOLD_CATEGORIES = [
    '200K - 500K', '500K - 1M', '1M - 2M', '2M - 5M', '5M - 10M',
    '10M - 20M', '20M - 50M', '50M - 100M', '100M - 200M'
]

## Hàm tải dữ liệu

In [ ]:
def load_data(file_path):
    df = pd.read_csv(file_path)

    df.columns = [
        'Name', 'Release date', 'Estimated owners', 'Peak CCU', 'Required age', 'Price',
        'Discount', 'DLC count', 'About the game', 'Supported languages',
        'Full audio languages', 'Reviews', 'Header image', 'Website', 'Support url',
        'Support email', 'Windows', 'Mac', 'Linux', 'Metacritic score',
        'Metacritic url', 'User score', 'Positive', 'Negative', 'Score rank',
        'Achievements', 'Recommendations', 'Notes', 'Average playtime forever',
        'Average playtime two weeks', 'Median playtime forever',
        'Median playtime two weeks', 'Developers', 'Publishers', 'Categories',
        'Genres', 'Tags', 'Screenshots', 'Movies'
    ]

    print(f'Tong so dong ban dau: {len(df):,}')
    print('So gia tri thieu theo cot:')
    print(df.isnull().sum().sort_values(ascending=False).head(10))
    return df

## Hàm làm sạch và hỗ trợ phân tích

In [ ]:
def clean_data(df):
    cleaned = df.copy()
    cleaned = cleaned[cleaned['Estimated owners'] != '0 - 0'].copy()
    cleaned['Estimated owners'] = cleaned['Estimated owners'].map(ESTIMATED_OWNERS_MAP)
    cleaned = cleaned[cleaned['Estimated owners'].notna()].copy()
    cleaned['Release date'] = pd.to_datetime(cleaned['Release date'], errors='coerce')
    cleaned['Release year'] = cleaned['Release date'].dt.year
    cleaned = cleaned[(cleaned['Release year'].isna()) | ((cleaned['Release year'] >= 2000) & (cleaned['Release year'] <= 2026))]

    numeric_columns = [
        'Price', 'Positive', 'Negative', 'Recommendations', 'Peak CCU',
        'Average playtime forever', 'Average playtime two weeks',
        'Median playtime forever', 'Median playtime two weeks'
    ]
    for column in numeric_columns:
        cleaned[column] = pd.to_numeric(cleaned[column], errors='coerce')

    for system in ['Windows', 'Mac', 'Linux']:
        cleaned[system] = cleaned[system].fillna(False).astype(bool)

    cleaned['About the game'] = cleaned['About the game'].fillna('')
    cleaned['Genres'] = cleaned['Genres'].fillna('Unknown')
    cleaned['Name'] = cleaned['Name'].fillna('Unknown')
    return cleaned


def summarize_cleaned_data(df):
    summary = pd.DataFrame({
        'chi_so': [
            'Tong so game sau lam sach',
            'So nhom estimated owners',
            'So game mien phi',
            'So game co review hop le',
            'So game pho bien (>= 200K owners)'
        ],
        'gia_tri': [
            len(df),
            df['Estimated owners'].nunique(),
            int((df['Price'] == 0).sum()),
            int(((df['Positive'] > 4) & (df['Negative'] > 4)).sum()),
            int(df['Estimated owners'].isin(POPULAR_THRESHOLD_CATEGORIES).sum())
        ]
    })
    return summary


def get_genre_count(df, to_df=False, column_name='Count'):
    genre_dict = {}
    for genres, number in df['Genres'].value_counts().items():
        for genre in str(genres).split(','):
            genre = genre.strip()
            if not genre:
                continue
            genre_dict[genre] = genre_dict.get(genre, 0) + number

    if to_df:
        genre_df = pd.DataFrame.from_dict(genre_dict, orient='index', columns=[column_name])
        return genre_df.sort_values(column_name, ascending=False)
    return genre_dict


def has_cjk_character(text):
    cjk_pattern = re.compile(r'[\u4E00-\u9FFF\u3040-\u309F\u30A0-\u30FF\uAC00-\uD7AF]')
    return bool(cjk_pattern.search(str(text)))

## Tải dữ liệu và xem kết quả làm sạch

In [ ]:
file_path = 'games.csv'
df_raw = load_data(file_path)
df_cleaned = clean_data(df_raw)

print(f'So dong sau lam sach: {len(df_cleaned):,}')
display(summarize_cleaned_data(df_cleaned))
display(df_cleaned.head())

## Hàm vẽ biểu đồ

In [ ]:
def plot_games_distribution(df):
    counts = df['Estimated owners'].value_counts().reindex(ESTIMATED_OWNERS_ORDER).fillna(0)
    ax = counts.plot.bar(color='steelblue')
    ax.set_title('Distribution of Games by Number of Estimated Owners')
    ax.set_xlabel('Estimated Owners')
    ax.set_ylabel('Number of games')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()


def plot_price_analysis(df):
    percent_free = (df[df['Price'] == 0]['Estimated owners'].value_counts() / df['Estimated owners'].value_counts() * 100).reindex(ESTIMATED_OWNERS_ORDER)
    paid_games = df[df['Price'] != 0]
    average_price = pd.concat([
        df.groupby('Estimated owners')['Price'].mean().reindex(ESTIMATED_OWNERS_ORDER),
        paid_games.groupby('Estimated owners')['Price'].mean().reindex(ESTIMATED_OWNERS_ORDER)
    ], axis=1)
    average_price.columns = ['Average price', 'Average price excluding free games']

    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    percent_free.fillna(0).plot.bar(ax=axes[0], color='coral')
    axes[0].set_title('Percentage of Free Games by Popularity Category')
    axes[0].set_xlabel('Estimated Owners')
    axes[0].set_ylabel('Percentage of free games (%)')
    axes[0].tick_params(axis='x', rotation=45)

    average_price.fillna(0).plot.bar(ax=axes[1])
    axes[1].set_title('Average Game Price by Popularity Category')
    axes[1].set_xlabel('Estimated Owners')
    axes[1].set_ylabel('Average price (USD)')
    axes[1].tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.show()


def plot_review_analysis(df):
    ratings = df[(df['Positive'] > 4) & (df['Negative'] > 4)][['Name', 'Estimated owners', 'Positive', 'Negative']].copy()
    ratings['Fraction positive'] = ratings['Positive'] / (ratings['Positive'] + ratings['Negative'])
    rating_average = ratings.groupby('Estimated owners')['Fraction positive'].mean().reindex(ESTIMATED_OWNERS_ORDER) * 100

    ax = rating_average.plot.bar(color='mediumseagreen')
    ax.set_title('Average Positive Review Percentage by Popularity Category')
    ax.set_xlabel('Estimated Owners')
    ax.set_ylabel('Average fraction of positive reviews (%)')
    ax.axhline(y=70, color='red', linestyle='--', alpha=0.6)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()


def plot_genre_heatmap(df):
    genre_fraction_per_owner = pd.DataFrame()
    for owners_category in ESTIMATED_OWNERS_ORDER:
        subset = df[df['Estimated owners'] == owners_category]
        if len(subset) == 0:
            continue
        new_column = get_genre_count(subset, to_df=True, column_name=owners_category) / len(subset)
        genre_fraction_per_owner = pd.concat([genre_fraction_per_owner, new_column], axis=1)

    genre_fraction_per_owner = genre_fraction_per_owner.fillna(0)
    top_12_genres = (get_genre_count(df, to_df=True) / len(df)).head(12).index

    plt.figure(figsize=(16, 10))
    sns.heatmap(genre_fraction_per_owner.loc[top_12_genres].fillna(0) * 100, annot=True, fmt='.1f', cmap='Blues', cbar_kws={'label': 'Percentage (%)'})
    plt.title('Top 12 Genres Distribution Across Popularity Categories (%)')
    plt.xlabel('Estimated Owners')
    plt.ylabel('Genre')
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_description_length(df):
    description_df = df[df['About the game'].ne('')].copy()
    description_df['Description length'] = description_df['About the game'].apply(lambda text: len(str(text).split()))
    description_df = description_df[~description_df['About the game'].apply(has_cjk_character)]
    description_length_average = description_df.groupby('Estimated owners')['Description length'].mean().reindex(ESTIMATED_OWNERS_ORDER)

    ax = description_length_average.plot.bar(color='mediumpurple')
    ax.set_title('Average Game Description Length by Popularity Category')
    ax.set_xlabel('Estimated Owners')
    ax.set_ylabel('Average number of words in game description')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()


def plot_os_support(df):
    system_support = pd.DataFrame({
        system: df.groupby('Estimated owners')[system].mean().reindex(ESTIMATED_OWNERS_ORDER) * 100
        for system in ['Windows', 'Mac', 'Linux']
    })

    ax = system_support.plot.bar(figsize=(14, 6))
    ax.set_title('Operating System Support by Popularity Category')
    ax.set_xlabel('Estimated Owners')
    ax.set_ylabel('Fraction of games (%)')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()


def plot_popular_genre_trends(df):
    popular_games = df[df['Estimated owners'].isin(POPULAR_THRESHOLD_CATEGORIES)].copy()
    popular_games = popular_games[popular_games['Release year'].between(2000, 2026, inclusive='both')].copy()

    years = sorted(popular_games['Release year'].dropna().unique())
    genre_trends = pd.DataFrame()
    genre_pct_trends = pd.DataFrame()

    for year in years:
        year_df = popular_games[popular_games['Release year'] == year]
        genre_count = get_genre_count(year_df, to_df=True, column_name=str(int(year)))
        genre_trends = pd.concat([genre_trends, genre_count], axis=1)
        genre_pct_trends = pd.concat([genre_pct_trends, genre_count / len(year_df) * 100], axis=1)

    genre_trends = genre_trends.fillna(0).T
    genre_pct_trends = genre_pct_trends.fillna(0).T
    top_genres = genre_trends.sum().sort_values(ascending=False).head(8).index

    fig, axes = plt.subplots(2, 1, figsize=(16, 14))
    for genre in top_genres:
        axes[0].plot(genre_trends.index.astype(int), genre_trends[genre], marker='o', label=genre, linewidth=2)
    axes[0].set_title('Genre Trends of Popular Games Over Years (200K+ Owners)')
    axes[0].set_xlabel('Release Year')
    axes[0].set_ylabel('Number of Games')
    axes[0].grid(True, alpha=0.3)
    axes[0].legend(title='Genre', bbox_to_anchor=(1.02, 1), loc='upper left')

    for genre in top_genres:
        axes[1].plot(genre_pct_trends.index.astype(int), genre_pct_trends[genre], marker='o', label=genre, linewidth=2)
    axes[1].set_title('Genre Percentage Trends of Popular Games Over Years (200K+ Owners)')
    axes[1].set_xlabel('Release Year')
    axes[1].set_ylabel('Percentage of Games (%)')
    axes[1].grid(True, alpha=0.3)
    axes[1].legend(title='Genre', bbox_to_anchor=(1.02, 1), loc='upper left')

    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(20, 8))
    sns.heatmap(genre_pct_trends[top_genres].T, annot=True, fmt='.1f', cmap='YlOrRd', cbar_kws={'label': 'Percentage (%)'})
    plt.title('Heatmap: Genre Trends of Popular Games (200K+ Owners) by Year')
    plt.xlabel('Release Year')
    plt.ylabel('Genre')
    plt.tight_layout()
    plt.show()

## Chạy các biểu đồ

In [ ]:
plot_games_distribution(df_cleaned)
plot_price_analysis(df_cleaned)
plot_review_analysis(df_cleaned)
plot_genre_heatmap(df_cleaned)
plot_description_length(df_cleaned)
plot_os_support(df_cleaned)
plot_popular_genre_trends(df_cleaned)